In [ ]:
# Setup: install Qiskit (runs automatically in Colab, no-op in Binder)
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen die Verwendung dieser oder neuerer Versionen.

```
qiskit[all]~=2.3.0
```
</details>
Falls Sie einen Satz von $n$ Bits (oder Qubits) haben, werden Sie normalerweise jedes Bit mit $0
\rightarrow n-1$ beschriften. Verschiedene Software und Ressourcen müssen entscheiden, wie sie
diese Bits sowohl im Computerspeicher als auch bei der Anzeige auf dem Bildschirm anordnen.

## Qiskit-Konventionen

So ordnet das Qiskit SDK Bits in verschiedenen Szenarien an.

### Quantenschaltkreise

Die `QuantumCircuit`-Klasse speichert ihre Qubits in einer Liste
(`QuantumCircuit.qubits`). Der Index eines Qubits in dieser Liste definiert die
Bezeichnung des Qubits.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Qubit

qc = QuantumCircuit(2)
qc.qubits[0]  # qubit "0"

Qubit(QuantumRegister(2, "q"), 0)

<Qubit register=(2, "q"), index=0>

### Circuit diagrams

On a circuit diagram, qubit $0$ is the topmost qubit, and qubit $n-1$ the
bottommost qubit. You can change this with the `reverse_bits` argument of
`QuantumCircuit.draw` (see [Change ordering in
Qiskit](#change-ordering-in-qiskit)).

In [2]:
qc.x(1)
qc.draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

### Schaltkreisdiagramme
In einem Schaltkreisdiagramm ist Qubit $0$ das oberste Qubit und Qubit $n-1$ das
unterste Qubit. Sie können dies mit dem `reverse_bits`-Argument von
`QuantumCircuit.draw` ändern (siehe [Reihenfolge in
Qiskit ändern](#change-ordering-in-qiskit)).

In [3]:
from qiskit.primitives import StatevectorSampler as Sampler

qc.measure_all()

job = Sampler().run([qc])
result = job.result()
print(f" > Counts: {result[0].data.meas.get_counts()}")

 > Counts: {'10': 1024}


### Strings

When displaying or interpreting a list of bits (or qubits) as a string, bit
$n-1$ is the leftmost bit, and bit $0$ is the rightmost bit. This is because we
usually write numbers with the most significant digit on the left, and in
Qiskit, bit $n-1$ is interpreted as the most significant bit.

For example, the following cell defines a `Statevector` from a string of
single-qubit states. In this case, qubit $0$ is in state $|+\rangle$, and
qubit $1$ in state $|0\rangle$.

In [4]:
from qiskit.quantum_info import Statevector

sv = Statevector.from_label("0+")
sv.probabilities_dict()

{np.str_('00'): np.float64(0.4999999999999999),
 np.str_('01'): np.float64(0.4999999999999999)}

### Ganzzahlen
Bei der Interpretation von Bits als Zahl ist Bit $0$ das niederwertigste Bit und
Bit $n-1$ das höchstwertige. Dies ist hilfreich beim Programmieren, da jedes Bit den
Wert $2^\text{label}$ hat (wobei label der Index des Qubits in
`QuantumCircuit.qubits` ist). Die folgende Schaltkreisausführung endet beispielsweise
damit, dass Bit $0$ `0` ist und Bit $1$ `1` ist. Dies wird als die
Dezimalzahl `2` interpretiert (gemessen mit Wahrscheinlichkeit `1.0`).

In [5]:
print(sv[1])  # amplitude of state |01>
print(sv[2])  # amplitude of state |10>

(0.7071067811865475+0j)
0j


### Gates

Each gate in Qiskit can interpret a list of qubits in its own way, but
controlled gates usually follow the convention `(control, target)`.

For example, the following cell adds a controlled-X gate where qubit $0$ is
the control and qubit $1$ is the target.

In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw()

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

### Zeichenketten
Bei der Anzeige oder Interpretation einer Liste von Bits (oder Qubits) als Zeichenkette ist Bit
$n-1$ das linkeste Bit und Bit $0$ das rechteste Bit. Dies liegt daran, dass wir
normalerweise Zahlen mit der höchstwertigen Ziffer links schreiben, und in
Qiskit wird Bit $n-1$ als das höchstwertige Bit interpretiert.

Die folgende Zelle definiert beispielsweise einen `Statevector` aus einer Zeichenkette von
Einzelqubit-Zuständen. In diesem Fall befindet sich Qubit $0$ im Zustand $|+\rangle$ und
Qubit $1$ im Zustand $|0\rangle$.

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.x(0)
qc.draw(reverse_bits=True)

          
q_1: ─────
     ┌───┐
q_0: ┤ X ├
     └───┘

You can use the `reverse_bits` method to return a new circuit with the
qubits' labels reversed (this does not mutate the original circuit).

In [8]:
qc.reverse_bits().draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Dies verursacht gelegentlich Verwirrung bei der Interpretation einer Bit-Zeichenkette, da Sie
möglicherweise erwarten, dass das linkeste Bit Bit $0$ ist, während es normalerweise Bit
$n-1$ darstellt.

### Zustandsvektor-Matrizen
Bei der Darstellung eines Zustandsvektors als Liste komplexer Zahlen (Amplituden)
ordnet Qiskit diese Amplituden so an, dass die Amplitude am Index $x$ den
Berechnungsbasiszustand $|x\rangle$ darstellt.